In [1]:
import pandas as pd

In [28]:
df=pd.read_csv("./customer_shopping_behavior.csv")

In [29]:
df.head(2)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly


In [30]:
df.isna().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [31]:
df.duplicated().sum()

np.int64(0)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [33]:
df[df['Review Rating'].isna()].head(2)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
38,39,29,Male,Dress,Clothing,37,Florida,M,Red,Winter,NaN,Yes,2-Day Shipping,Yes,Yes,44,Venmo,Every 3 Months
50,51,49,Male,Blouse,Clothing,28,Maryland,M,Red,Spring,NaN,Yes,Store Pickup,Yes,Yes,39,Debit Card,Bi-Weekly


In [34]:
# df[df['Review Rating'].isna()]['Item Purchased'].value_counts()# well that is the best to use the Item Purchased but we will use the category intead 

In [35]:
# in order to fill the null values we use either the median or mean but it preferd to use the median because it robust to outliers
# df['Review Rating'].fillna(df['Review Rating'].mean(),inplace=True)

In [36]:
df['Review Rating']=df.groupby('Category')['Review Rating'].transform(lambda x:x.fillna(x.median()))# x in here is a dataframe and its grouped by the category

In [37]:
df.isna().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [38]:
# for col in df.columns:
#     newname=col.lower().replace(" ","_")
#     df.rename(columns={col:newname},inplace=True)
# this is working but i will try more profitional way

In [39]:
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(" ","_")

In [40]:
df=df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

## feature engenring

In [41]:
#1 age groupes

In [42]:
groups=['Teenager','Adult','Middle_age','Old']

In [43]:
df['age_groups']=pd.qcut(
    df['age'],
    q=4,
    labels=['Teenager','Adult','Middle_age','Old']
)

In [44]:
#2 purchest_frequency_days

In [45]:
frequency_mapping={
    'Every 3 Months':90,
    'Annually':365,
    'Quarterly':90,
    'Monthly':30,
    'Bi-Weekly':14,
    'Fortnightly':14,
    'Weekly':7
}

In [46]:
df['purchest_frequency_days']=df['frequency_of_purchases'].map(frequency_mapping)

In [47]:
df['frequency_of_purchases'].value_counts()

frequency_of_purchases
Every 3 Months    584
Annually          572
Quarterly         563
Monthly           553
Bi-Weekly         547
Fortnightly       542
Weekly            539
Name: count, dtype: int64

In [48]:
(df['discount_applied']==df['promo_code_used']).all() # if they are true means we do not need both of those columns

np.True_

In [49]:
df.drop('promo_code_used',axis=1,inplace=True)

### after we done from the data explorition and data preprocessing we will export this data to sql to do some data analytic and answer some questions

In [51]:
# df.to_csv("customer_shopping_behavior_cleaned.csv")

In [53]:
# pip install sqlalchemy pymysql

In [54]:
from sqlalchemy import create_engine

In [55]:
username="root"
password="user"
host="localhost"
port="3306"
database="customer_behivior"
engine=create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
table_name='customer'
v=df.to_sql(table_name,engine,if_exists="replace",index=False)
if v:
    print("data successfully loaded")

data successfully loaded
